# Skeleton Notebook — Time Series Diagnostics (Practice From Scratch)

This is the **practice version** of `02_Extended_Lab_TimeSeries_Diagnostics.ipynb`. The narrative and structure are kept, but the code is left for you to write. Work through it top to bottom without peeking at the solutions notebook (`06_Solutions_TimeSeries_Diagnostics.ipynb`) — use it only to check your work afterward.

**Topics covered:** mean/variance/ergodicity, white noise & Ljung-Box, ACF/PACF, cross-correlation (CCF) & lead/lag detection, stationarity (visual + ADF/KPSS), differencing & seasonal differencing, capstone mini-project.

Make sure `airline-passengers.csv` is in the same folder as this notebook before running Part E/F.


## 0. Setup
Import the libraries you'll need: `numpy`, `pandas`, `statsmodels.api`, `matplotlib.pyplot`, `seaborn`, and from `statsmodels`: `plot_acf`, `plot_pacf`, `acorr_ljungbox`, `adfuller`, `kpss`, `seasonal_decompose`. Also import `correlate` from `scipy.signal`.

In [ ]:
# TODO: imports


## Part A — Mean, Variance, and Ergodicity

**Task A1:** Simulate two 600-point series:
- `ar1`: a stationary AR(1) process, $x_t = 0.6\,x_{t-1} + \varepsilon_t$, $\varepsilon_t \sim N(0,1)$
- `regime`: white noise with standard deviation 1 for the first half of the series and standard deviation 4 for the second half

Plot both. Then compute and print the mean and standard deviation of each series' first half vs. second half.


In [ ]:
# TODO: Task A1


**Task A2 (🧪 extended exercise):** Simulate a process whose **mean** shifts partway through (e.g., add a step function to noise: first half centered at 0, second half centered at 5). Compute the mean of the first third, middle third, and last third of the series. Write a one-sentence conclusion about whether the full-sample mean is a trustworthy forecast of the next value.


In [ ]:
# TODO: Task A2


## Part B — White Noise & the Ljung-Box Test

**Task B1:** Generate 1,000 samples of standard normal white noise (`random_white_noise`). Plot the raw series with a horizontal reference line at 0, and its ACF (use `plot_acf`), side by side. Then run `acorr_ljungbox` with `lags=[10, 30, 50]` and interpret the p-values in a short comment.


In [ ]:
# TODO: Task B1


**Task B2:** Using the `ar1` series from Part A, plot it and its ACF, then run the Ljung-Box test at `lags=[10, 30, 50]`. State (in a comment or markdown cell) whether you reject or fail to reject the null hypothesis of no autocorrelation, and explain why that result makes sense given how `ar1` was generated.


In [ ]:
# TODO: Task B2


**Task B3 (🧪 extended):** Generate white noise with `size=30`, `size=200`, and `size=2000`. For each, run `acorr_ljungbox(..., lags=[10])`. Repeat with 3-4 different random seeds. Do you ever see a p-value below 0.05 for data you *know* is white noise? What does this imply about relying on one hypothesis test at one lag?


In [ ]:
# TODO: Task B3


## Part C — Autocorrelation (ACF) and Partial Autocorrelation (PACF)

**Task C1:** Load the `realinv` and `realdpi` columns from `statsmodels`' macrodata (`sm.datasets.macrodata.load().data`), cast both to `float32` and round to 2 decimals, and store as `df_mod`.


In [ ]:
# TODO: Task C1


**Task C2:** For `realinv`, create a 2x3 grid of plots: original data / original ACF (lags=50) / original PACF (lags=50) on the top row, and first-differenced data / differenced ACF / differenced PACF on the bottom row. Use `np.diff(x, n=1)` for differencing.


In [ ]:
# TODO: Task C2


**Task C3 (🧪 extended):** Repeat Task C2 for `realdpi`. Compare the ACF/PACF shapes to `realinv` — similar or different candidate AR/MA orders?


In [ ]:
# TODO: Task C3


**Task C4 (🧪 extended — over-differencing):** Apply a *second* first-order difference to `realinv` (difference the already-differenced series again). Plot its ACF/PACF. Look for a strong negative spike at lag 1 — a classic sign of over-differencing.


In [ ]:
# TODO: Task C4


## Part D — Cross-Correlation (CCF)

**Task D1:** Write a function `plot_ccf(data_a, data_b, lag_lookback, percentile=95)` that:
1. Computes the mean-centered, normalized cross-correlation using `scipy.signal.correlate` divided by `std(a) * std(b) * n`
2. Slices out `lag_lookback` lags on either side of 0
3. Plots a stem plot with confidence bands at $\pm z/\sqrt{n}$ (z = 1.645, 1.96, or 2.576 for 90/95/99%)
4. Draws a vertical line at lag 0


In [ ]:
# TODO: Task D1 — write plot_ccf()


**Task D2:** Build `df_diff` with the first-differenced `realinv` and `realdpi` series. Call `plot_ccf(df_diff['realdpi'], df_diff['realinv'], lag_lookback=50)`. Where is the peak? What does that tell you about lead/lag?


In [ ]:
# TODO: Task D2


**Task D3 (🧪 extended — synthetic leading indicator):** Simulate `ad_spend` (300 points of standard normal noise) and `sales = 0.8 * shift(ad_spend, 2) + noise`, where `sales` responds to `ad_spend` from 2 periods earlier (use `np.roll` and fix the wrap-around at the start). Plot the CCF between `ad_spend` and `sales`. Confirm the peak lands at lag 2. Then use `pandas.Series.shift()` to align `ad_spend` forward by 2 and re-run the CCF — the peak should now be at lag 0.


In [ ]:
# TODO: Task D3


## Part E — Stationarity

**Task E1:** Load `airline-passengers.csv` (`index_col=0`), parse the index as dates with `pd.to_datetime(..., format='%Y-%m')`, and plot the series.


In [ ]:
# TODO: Task E1


**Task E2:** Run `seasonal_decompose` on the series and plot the result. Also create a boxplot of `Passengers` grouped by `data.index.year` using `seaborn.boxplot`. What do the trend/seasonal/residual panels and the year-by-year boxplots tell you about stationarity?


In [ ]:
# TODO: Task E2


**Task E3:** Plot the ACF of the raw series (`lags=20`) and run `acorr_ljungbox(data, lags=[50], return_df=True)`. Then write a function:

```python
def stationarity_report(series, name="series"):
    # run adfuller() and kpss() on `series`
    # print the test statistic and p-value for each
    # print a plain-language conclusion for each test
```

Apply it to the raw series and to the first-differenced series.


In [ ]:
# TODO: Task E3


**Task E4 (🧪 extended — log + seasonal differencing):** Compute `log_passengers = np.log(data['Passengers'])`. Then compute a seasonal difference (`.diff(12)`) of the log series, and a further first-order difference of that (`.diff(1)`). Plot all three transformed series, and run `stationarity_report` on each. What is the effect of the log transform on the growing seasonal amplitude? What does the seasonal difference remove that a plain first difference does not?


In [ ]:
# TODO: Task E4


## Part F — Capstone Mini-Project

**Task F1:** Simulate three 400-point series:
- `series1`: pure white noise
- `series2`: a random walk (cumulative sum of white noise)
- `series3`: a stationary AR(2) (`0.5*x[t-1] - 0.3*x[t-2] + noise`) plus a seasonal sine wave of period 12 and amplitude 3

Plot all three.


In [ ]:
# TODO: Task F1


**Task F2:** For each of `series1`, `series2`, `series3`:
1. Plot the ACF/PACF
2. Run the Ljung-Box test
3. Run `stationarity_report` (ADF + KPSS)
4. If non-stationary, apply the appropriate transform and re-test
5. Write a 2-3 sentence verdict: is it white noise / trend-stationary / difference-stationary / seasonal-non-stationary? What transform would you use before modeling? What AR/MA order does the (transformed) ACF/PACF suggest?


In [ ]:
# TODO: Task F2


## Wrap-up Questions (answer in markdown cells below)

1. Why check for white noise before fitting an AR/MA model?
2. Why does a trending series' ACF decay slowly, and why does differencing fix that?
3. What does a CCF peak at a positive lag tell you about which series leads?
4. Give an example where ADF and KPSS might disagree, and the follow-up you'd do.
5. Why might differencing alone be insufficient for a series like Air Passengers?


In [ ]:
# Your answers here (markdown or comments)
